In [ ]:
%load_ext autoreload
%autoreload 2

# Eva Data

This notebook builds a `MultiDataset` containing exactly one `ZarrDataset`, loads one batch, visualizes one image with `mediapy`, and prints the rest of the batch.

In [ ]:
from pathlib import Path

import cv2
import imageio_ffmpeg
import mediapy as mpy
import numpy as np
import torch

from egomimic.rldb.embodiment.eva import Eva
from egomimic.rldb.embodiment.human import Human
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, ZarrDataset
from egomimic.rldb.zarr.zarr_dataset_multi import S3EpisodeResolver
from egomimic.rldb.filters import DatasetFilter
from egomimic.utils.aws.aws_data_utils import load_env

# Ensure mediapy can find an ffmpeg executable in this environment
mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())

In [ ]:
TEMP_DIR = "/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest/" # replace with your own temp directory for caching S3 data
load_env()

In [ ]:
key_map = Eva.get_keymap(keymap_mode="cartesian")
transform_list = Eva.get_transform_list(action_mode="cartesian", coord_frame="camframe", rotation_mode="euler")

resolver = S3EpisodeResolver(
    TEMP_DIR, key_map=key_map, transform_list=transform_list
)
filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-01-03-02-16-02-519000'}"
    ]
)
multi_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(multi_ds, batch_size=1, shuffle=False)

In [ ]:
# Separate YPR visualization preview
for batch in loader:
    vis_ypr = Eva.viz_transformed_batch(batch, mode="axes")
    mpy.show_image(vis_ypr)
    break

In [ ]:
images = []
for i, batch in enumerate(loader):
    vis = Eva.viz_transformed_batch(batch, mode="traj+rotation")
    images.append(vis)
    if i > 10:
        break

mpy.show_video(images, fps=30)

## Human Datasets
All human data uses the single `Human` embodiment. Per-source differences (head pose, action stride) are passed as args: `Human.get_keymap(keymap_mode=..., has_head_pose=...)`, `Human.get_transform_list(action_mode=..., coord_frame=..., rotation_mode=..., stride=...)`.

In [ ]:
from egomimic.rldb.zarr.zarr_dataset_multi import LocalEpisodeResolver

key_map = Human.get_keymap(keymap_mode="cartesian")
transform_list = Human.get_transform_list(action_mode="cartesian", coord_frame="camframe", rotation_mode="euler", stride=3)

TEMP_DIR = '/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest/'
resolver = LocalEpisodeResolver(
    TEMP_DIR,
    key_map=key_map,
    transform_list=transform_list,
)

filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-05-01-02-52-58-000000'}"
    ]
)

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=False, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
batch = next(iter(loader))

In [ ]:
batch.keys()

In [ ]:
ims = []
for i, batch in enumerate(loader):
    vis = Human.viz_transformed_batch(batch, mode="traj")
    ims.append(vis)
    if i > 10:
        break

mpy.show_video(ims, fps=30)

In [ ]:
# Human YPR video (same data loop, YPR overlay)
ims_ypr = []
for i, batch in enumerate(loader):
    vis_ypr = Human.viz_transformed_batch(batch, mode="axes")
    ims_ypr.append(vis_ypr)
    if i > 20:
        break

mpy.show_video(ims_ypr, fps=30)

In [ ]:
key_map = Human.get_keymap(keymap_mode="keypoints")
transform_list = Human.get_transform_list(action_mode="keypoints", coord_frame="camframe", rotation_mode="euler", stride=3)

resolver = S3EpisodeResolver(
    TEMP_DIR,
    key_map=key_map,
    transform_list=transform_list,
)

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
from egomimic.rldb.embodiment.human import _build_human_keypoints_revert_eef_frame_transform_list
ims_keypoints = []
revert_transform_list = _build_human_keypoints_revert_eef_frame_transform_list(is_quat=False)
for i, batch in enumerate(loader):
    vis_keypoints = Human.viz_transformed_batch(batch, mode="keypoints", viz_batch_key="actions_keypoints")
    ims_keypoints.append(vis_keypoints)
    if i > 20:
        break

mpy.show_video(ims_keypoints, fps=30)

## Aria Gaze

In [ ]:
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, ZarrDataset
import mediapy as mpy
import numpy as np
import torch
from pathlib import Path

# NOTE: a dedicated "gaze" keymap mode is not implemented on Human; gaze viz
# (Human.viz mode="gaze") is aria-MPS-specific and consumes obs_eye_gaze.
keymap_gaze = Human.get_keymap(keymap_mode="gaze")

filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-05-01-02-52-58-000000'}"
    ]
)
resolver = S3EpisodeResolver(
    TEMP_DIR, key_map=keymap_gaze
)
cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
loader

In [ ]:
batch = next(iter(loader))
print(batch.keys())

In [ ]:
ims_gaze = []
for i, batch in enumerate(loader):
    vis_gaze = Human.viz_transformed_batch(
        batch,
        mode="gaze",
        viz_batch_key="eye_gaze",
        image_key="images.front_1",
    )
    ims_gaze.append(vis_gaze)
    if i > 20:
        break
mpy.show_video(ims_gaze, fps=30)

# Annotation Viz

In [ ]:
from egomimic.rldb.filters import DatasetFilter

# Scale-style data has no head pose -> has_head_pose=False; annotations via annotation_key.
key_map = Human.get_keymap(
    keymap_mode="cartesian", has_head_pose=False, annotation_key="annotations"
)
transform_list = None

resolver = S3EpisodeResolver(
    folder_path = "/coc/flash7/scratch/egoverseDebugDatasets/egoverseS3DatasetTest/",
    key_map=key_map,
    transform_list=transform_list,
)

filters = DatasetFilter(
    filter_lambdas=[
        "lambda row: row['episode_hash'] in {'2026-03-16-01-22-26-448000'}"
    ]
)

cloudflare_ds = MultiDataset._from_resolver(
    resolver, filters=filters, sync_from_s3=True, mode="total"
)

loader = torch.utils.data.DataLoader(cloudflare_ds, batch_size=1, shuffle=False)

In [ ]:
batch = next(iter(loader))
batch['annotations']

In [ ]:
ims_annotations = []
for i, batch in enumerate(loader):
    vis = Human.viz_transformed_batch(batch, mode="annotations", viz_batch_key="annotations")
    ims_annotations.append(vis)
    if i > 10:
        break
mpy.show_video(ims_annotations, fps=30)

# ABC Data

An ABC-130k episode on the two-arm YAM station, pulled from R2 with
`S3EpisodeResolver` and loaded with the `Yam` embodiment (bimanual, parallel-jaw,
like `Eva`).

`Yam.EXTRINSICS` is the top-camera transform, so `coord_frame="camframe"` gives
camera-frame poses and the overlays work as they do for Eva. ABC's MCAP records
no extrinsics; this one is composed from the published station model
(`i2rt-robotics/i2rt`, `robot_models/station/yam_station_*_4310_d405`) and refined
to the real mount. It covers the **RealSense D405** stations; episodes on the
wider camera carry intrinsics only, so use an `eef_frame` coord_frame there.

`S3EpisodeResolver` filters against the **SQL episode table** (not local zarr
attrs), and syncs anything missing into `ABC_DATA_DIR`. Episodes already there
are skipped, so this is cheap on a warm cache.

This section is self-contained: it does not depend on the Eva/Human cells above.

In [ ]:
import json
from pathlib import Path

import imageio_ffmpeg
import mediapy as mpy
import numpy as np
import torch
import zarr

from egomimic.rldb.embodiment.yam import Yam
from egomimic.rldb.filters import DatasetFilter
from egomimic.rldb.zarr.zarr_dataset_multi import MultiDataset, S3EpisodeResolver
from egomimic.utils.aws.aws_data_utils import load_env

mpy.set_ffmpeg(imageio_ffmpeg.get_ffmpeg_exe())
load_env()  # R2 + DB credentials from ~/.egoverse_env

# Downloads land here. Same folder the local conversions live in, so episodes
# already present are reused rather than re-fetched: the resolver looks for a
# directory named exactly <episode_hash>, and the converter's <episode_hash>.zarr
# stores carry a <episode_hash> symlink alongside them.
ABC_DATA_DIR = "/coc/flash7/scratch/acheluva3/abc_data/zarr"

# The only thing to change to look at a different episode. It is the filter the
# resolver runs against the SQL episode table, and the resolver syncs it down if
# it is not already in ABC_DATA_DIR.
EPISODE_HASH = "c7b01f60-b438-41a2-be41-a2bbb5c0bd0a"
print("episode:", EPISODE_HASH)

## Subtask annotations

ABC ships subtask labels in a sibling `annotation.mcap`; the converter maps them to
frame ranges in the `annotations` array.

In [ ]:
key_map = Yam.get_keymap(keymap_mode="cartesian", annotation_key="annotations")
# cartesian + camframe + euler = camera frame, 14D [L xyz ypr g, R xyz ypr g]
# â the layout the viz helpers expect, matching Eva.
transform_list = Yam.get_transform_list(action_mode="cartesian", coord_frame="camframe", rotation_mode="euler")

# The overlays only draw on the front camera, so drop the wrist streams: one JPEG
# decode per sample instead of three, which matters across a multi-thousand-frame
# episode.
viz_key_map = {
    k: v
    for k, v in key_map.items()
    if v["zarr_key"] not in ("images.left_wrist", "images.right_wrist")
}

# Filters run against the SQL episode table here, so the fields are app.episodes
# columns rather than the zarr attrs LocalEpisodeResolver matches on. To pull a
# whole task instead:
#   "lambda row: row['lab']=='abc' and row['task']=='fold and stack the skirts'"
filters = DatasetFilter(
    filter_lambdas=[f"lambda row: row['episode_hash'] == '{EPISODE_HASH}'"]
)

# sync_from_s3=True downloads the episode if ABC_DATA_DIR does not already have it.
abc_ds = MultiDataset._from_resolver(
    S3EpisodeResolver(
        ABC_DATA_DIR, key_map=viz_key_map, transform_list=transform_list
    ),
    filters=filters,
    sync_from_s3=True,
    mode="total",
)

# Only now is the store guaranteed on disk. <hash> resolves either to a directory
# the resolver just synced or to the symlink beside a local <hash>.zarr.
store = zarr.open(str(Path(ABC_DATA_DIR) / EPISODE_HASH), mode="r", zarr_format=3)
T, FPS = store.attrs["total_frames"], store.attrs["fps"]
print(f"{store.attrs['task_name']!r} | {T} frames @ {FPS}fps | {store.attrs['embodiment']}")

# The videos below render only the first PREVIEW_SECONDS at the capture rate.
PREVIEW_SECONDS = 30
PREVIEW_FRAMES = min(len(abc_ds), int(PREVIEW_SECONDS * FPS))

loader = torch.utils.data.DataLoader(abc_ds, batch_size=1, shuffle=False)
print(f"{len(abc_ds)} samples | preview = first {PREVIEW_FRAMES} frames ({PREVIEW_SECONDS}s)")

## Subtask annotations

ABC ships subtask labels in a sibling `annotation.mcap`; the converter maps them to
frame ranges in the `annotations` array.

In [ ]:
for a in (json.loads(b) for b in store["annotations"][:]):
    span = (a["end_idx"] - a["start_idx"]) / FPS
    print(f"  {a['start_idx']:>5} -> {a['end_idx']:<5} ({span:5.1f}s)  {a['text']}")

In [ ]:
# Separate YPR visualization preview
for batch in loader:
    vis_ypr = Yam.viz_transformed_batch(batch, mode="axes")
    mpy.show_image(vis_ypr)
    break

In [ ]:
images = []
for i, batch in enumerate(loader):
    vis = Yam.viz_transformed_batch(batch, mode="traj+rotation")
    images.append(vis)
    if i + 1 >= PREVIEW_FRAMES:
        break

mpy.show_video(images, fps=int(FPS))


## Action-overlay video

`Yam.get_revert_transform_list()` is the wrist-frame counterpart of Eva's
`_build_eva_bimanual_revert_eef_frame_transform_list`: it undoes the wrist-frame
step (`inverse=False`, i.e. re-composes the chunk onto the obs pose) and hands
back absolute poses.

**Why this is not drawn on the image like the Eva/Human sections.** Those project
actions through `K` with `_viz_traj`, which is only meaningful when the poses are
in the *camera* frame. Eva gets that from `Eva.EXTRINSICS`. ABC records no
extrinsics, so the revert lands in the *station world* frame and there is nothing
to project with. Checked, and none of these supply one:

- the MCAP `<topic>-info` `CameraCalibration` messages carry K, resolution and a
  distortion model only;
- the YAM MJCF the dataset ships defines no cameras;
- ABC's own `export_mcap.py` trains on joint space and `viz_policy.py` is a 3D
  Viser view of a sim rollout, so upstream never projects actions either.

Fitting the camera pose from the data (PnP against motion centroids) was tried and
rejected: cloth motion dominates the centroids, and the reprojected gripper
positions visibly miss the grippers.

So the action chunk is drawn in a station top-down panel next to the frame. If you
ever obtain a world-to-camera transform, set `Yam.EXTRINSICS` and the ordinary
`Yam.viz_transformed_batch(batch, mode="traj")` path starts working.

In [ ]:
# The revert transform: wrist-frame deltas -> absolute world poses.
sample = yam_ds[600]
print("wrist-frame action chunk:", np.asarray(sample["actions_cartesian"]).shape)

for tf in Yam.get_revert_transform_list(is_quat=False):
    sample = tf.transform(sample)
reverted = np.asarray(sample["actions_cartesian"])
print("reverted to world frame :", reverted.shape)

# Reverting must reproduce the commanded poses the converter wrote.
print("reverted L xyz[0]:", reverted[0, :3].round(4).tolist())
print("zarr     L xyz[600]:", store["left.cmd_ee_pose"][600, :3].round(4).tolist())

In [ ]:
import cv2

CHUNK = 100
PW = PH = 480

Lo = store["left.obs_ee_pose"][:T, :3];  Ro = store["right.obs_ee_pose"][:T, :3]
Lc = store["left.cmd_ee_pose"][:T, :3];  Rc = store["right.cmd_ee_pose"][:T, :3]
Lg = store["left.obs_gripper"][:T, 0];   Rg = store["right.obs_gripper"][:T, 0]

_all = np.vstack([Lo, Ro])
_mn = _all.min(0)[:2] - 0.05
_mx = _all.max(0)[:2] + 0.05

def to_px(xy):
    """World (X fwd, Y left) -> panel pixels, oriented like the top camera."""
    u = (xy[..., 0] - _mn[0]) / (_mx[0] - _mn[0])
    v = (xy[..., 1] - _mn[1]) / (_mx[1] - _mn[1])
    return np.stack([(1 - v) * (PW - 1), (1 - u) * (PH - 1)], -1).astype(np.int32)

_BG = np.full((PH, PW, 3), 18, np.uint8)
for _p, _c in ((Lo, (34, 52, 34)), (Ro, (52, 34, 34))):
    cv2.polylines(_BG, [to_px(_p[:, :2]).reshape(-1, 1, 2)], False, _c, 1, cv2.LINE_AA)
cv2.putText(_BG, "station top-down (X fwd up, Y left left)", (8, PH - 12),
            0, 0.42, (130, 130, 130), 1, cv2.LINE_AA)
cv2.putText(_BG, "bright = action chunk, dot = current EEF", (8, PH - 28),
            0, 0.42, (130, 130, 130), 1, cv2.LINE_AA)

def action_panel(t):
    p = _BG.copy()
    for cmd, obs, grip, col, nm, x0 in (
        (Lc, Lo, Lg, (80, 255, 80), "L", 10),
        (Rc, Ro, Rg, (80, 80, 255), "R", 60),
    ):
        chunk = to_px(cmd[t : min(t + CHUNK, T), :2])
        if len(chunk) > 1:
            cv2.polylines(p, [chunk.reshape(-1, 1, 2)], False, col, 2, cv2.LINE_AA)
            cv2.circle(p, tuple(chunk[-1]), 4, col, 2, cv2.LINE_AA)
        cv2.circle(p, tuple(to_px(obs[t, :2])), 7, col, -1, cv2.LINE_AA)
        h = int(100 * float(grip[t]))
        cv2.rectangle(p, (x0, 10), (x0 + 30, 110), (90, 90, 90), 1)
        cv2.rectangle(p, (x0, 110 - h), (x0 + 30, 110), col, -1)
        cv2.putText(p, nm, (x0 + 8, 126), 0, 0.45, col, 1, cv2.LINE_AA)
    cv2.putText(p, "grip", (10, 142), 0, 0.4, (130, 130, 130), 1, cv2.LINE_AA)
    return p

def overlay_frame(t):
    img = simplejpeg.decode_jpeg(store["images.front_1"][t : t + 1][0]).copy()
    cv2.rectangle(img, (0, 0), (img.shape[1] - 1, 44), (0, 0, 0), -1)
    cv2.putText(img, frame_label[t] or "-", (10, 20), 0, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(img, f"t={t}  {t / FPS:5.1f}s", (10, 38), 0, 0.45, (180, 180, 180), 1, cv2.LINE_AA)
    return np.hstack([img, action_panel(t)])

mpy.show_video([overlay_frame(t) for t in range(0, T, STEP)],
               fps=FPS / STEP, title="YAM: camera + action chunk")